In [ ]:
import os
import subprocess

MODEL_NAME = 'glot500'
SRC_LANG = 'hsb'
TRG_LANG = 'de'
BUCC_SETS = ['train', 'test']

MODEL_MAP = {
    'xlmr': 'xlm-roberta-base',
    'labse': 'sentence-transformers/LaBSE',
    'glot500': 'cis-lmu/glot500-base'
}
MODEL_PATH_HF = MODEL_MAP[MODEL_NAME]

#
DATA_DIR = '../data'
RESULTS_DIR = f'../all_executions/re-computation_folder/results_full_{MODEL_NAME}'
MINING_DIR = os.path.join(RESULTS_DIR, 'mining', 'bucc2017', f'{SRC_LANG}-{TRG_LANG}')

print(f"Starting SimAlign Post-Processing for model: {MODEL_NAME}")
print("-" * 50)


for data_split in BUCC_SETS:
    
    # Input file (main pipeline)
    mapping_file = f"{MINING_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim.pred"
    
    # original text files
    src_file = f"{DATA_DIR}/bucc_style_data/{SRC_LANG}-{TRG_LANG}/{SRC_LANG}-{TRG_LANG}.{data_split}.{SRC_LANG}"
    trg_file = f"{DATA_DIR}/bucc_style_data/{SRC_LANG}-{TRG_LANG}/{SRC_LANG}-{TRG_LANG}.{data_split}.{TRG_LANG}"
    gold_file = f"{DATA_DIR}/bucc_style_data/{SRC_LANG}-{TRG_LANG}/{SRC_LANG}-{TRG_LANG}.{data_split}.gold"
    
    # new output files 
    
    output_file_postprocessing = f"{MINING_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim.pred.postprocessing_5.5.3_excl_punctuation"
    output_file_postprocessing_res = f"{MINING_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim.pred.postprocessing.res_5.5.3_excl_punctuation"

    
    command = (
        f"python ../scripts/post-processing_simalign.py "
        f"--mapping-file {mapping_file} "
        f"--src-file {src_file} "
        f"--trg-file {trg_file} "
        f"--model-path {MODEL_PATH_HF} "
        f"--output-file {output_file_postprocessing} "
        f"--window-size 7 "
        f"--min-segment-length 0.35 "
        f"--segment-threshold 0.2 "
        f"--filtering-threshold 0.38 "
        f"--matching-method inter "     # inter, itermax, mwmf
        f"--token-type bpe "            # bpe, word
        f"--mean-subtraction "
        f"--filter-stopwords-trg "
        # f"--filter-stopwords-src "
        f"--exclude-punctuation "
    )
    
    print(f"Running SimAlign Post-Processing on {data_split} set...")
    subprocess.run(command, shell=True, check=True)
    
    eval_command = f"python ../code/scripts/bucc_f-score.py -p {output_file_postprocessing} -g {gold_file} > {output_file_postprocessing_res}"
    subprocess.run(eval_command, shell=True, check=True)
    
    with open(output_file_postprocessing_res, 'r') as f:
        print(f.read())

print("\n\nSimAlign Post-Processing Pipeline complete!")